In [1]:
import numpy as np
import pandas as pd
import re 
import sklearn
import html

In [2]:
file0 = "WELFake_Dataset.csv"
df_ = pd.read_csv(file0)
df_raw = df_.drop(columns=[df_.columns[0]])
#df_raw.head()

#print(len(df_raw['label']))
#print(list(df_raw.columns))

In [3]:
import nltk
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/raajjyoti/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
r_stoplist = set(nltk.corpus.stopwords.words("english"))

In [6]:
print(r_stoplist)

{"he'll", 'on', 'mustn', 'for', 'needn', 'we', 'they', 'his', 'll', 'ma', 'y', "won't", 'all', 'under', "couldn't", 'ours', 'won', "he's", 'if', 'haven', 'own', "i'd", 'she', "she's", 'when', 'ourselves', 'before', 'have', 'this', 'didn', 'having', 'is', 'you', 'or', 'into', 'am', 'doing', 'too', "it's", 'isn', 'up', "you'd", 'shouldn', 'ain', 'those', 'with', 'were', 'them', 'are', 'at', 'than', 'an', 'him', 'any', 'most', "you're", 'now', 'what', "we'll", "haven't", 'in', 're', 'will', 'further', 'about', 'mightn', 'off', 'being', 'between', 'by', 'more', "don't", "she'd", 'should', 't', 'that', 's', "mustn't", 'just', 'only', 'such', "it'd", 'then', 'shan', "she'll", 'don', 'from', "it'll", "you've", 'here', 'your', 'hasn', 'their', 'be', 'themselves', 'during', 'each', 'out', 'and', 'aren', "they've", 'how', 'my', "that'll", 'few', 'herself', 'down', 'himself', 'o', 'it', 'as', 'hadn', 'wasn', "they're", "should've", 'while', "shouldn't", 'these', "you'll", "i'm", 'me', 'd', 'not',

In [8]:
news_sw = ["said", "says", "told", "according", "reported", "reuters", "getty",
    "images", "image", "photo", "video", "watch", "read", "featured",
    "via", "twitter", "facebook", "post", "article", "news", "year",
    "years", "time", "day", "today", "new"]

stoplist = {word.replace("'","") for word in r_stoplist}.union(news_sw)
stoplist.discard("ill")

print(stoplist) 

{'mightnt', 'wouldnt', 'on', 'mustn', 'for', 'well', 'featured', 'itd', 'needn', 'we', 'they', 'id', 'his', 'll', 'ma', 'y', 'all', 'facebook', 'under', 'shouldnt', 'ours', 'according', 'won', 'youve', 'if', 'neednt', 'years', 'haven', 'own', 'she', 'when', 'image', 'ourselves', 'before', 'have', 'this', 'youll', 'photo', 'didn', 'having', 'is', 'you', 'or', 'into', 'am', 'theyre', 'doing', 'reuters', 'too', 'news', 'dont', 'isn', 'up', 'shouldn', 'wed', 'ain', 'those', 'didnt', 'with', 'video', 'were', 'them', 'are', 'at', 'hadnt', 'youd', 'than', 'an', 'him', 'any', 'most', 'im', 'now', 'what', 'will', 'in', 're', 'isnt', 'further', 'doesnt', 'about', 'article', 'mightn', 'off', 'being', 'between', 'by', 'more', 'should', 't', 'hed', 'werent', 'that', 'hell', 's', 'just', 'today', 'only', 'such', 'then', 'shan', 'don', 'from', 'here', 'your', 'hasn', 'their', 'be', 'hasnt', 'shant', 'themselves', 'getty', 'during', 'said', 'each', 'out', 'and', 'aren', 'how', 'my', 'few', 'wont', 'he

#for LDA the pipeline implmented would be as following:
- first lowercase everything, then URL/HTML tags removed, then punctuations and standalone numbers, and useless alphanums like 2016Trump or 20ea have been divided and standalone numbers have been removed 
- alphanums of form covid19, g20, ww3, etc have been preserved 
- now we have to remove stopwords for LDA (Latent Dirichlet Allocation) topic  modelling 
- CountVectorizer used to filter out the stopwords occuring in more than 85% and less than 15% of documents as it gives a sparse matrix of vectors that rely only on frequency of the features in the corpus. 
- more on countvectorizer in later cells  

#the usual pipeline for data preprocessing for NLP classification would be the lowercase corpus, remove stopwords, apply lemmatization/stemming then CountVectorizer

In [5]:
df = (
        df_raw
        #drop row only if nan exists in both title and text with how=all 
        .dropna(subset=['title', 'text'], how='all')
        #drop row if label=nan 
        .dropna(subset=['label'])
        #fill rest left nan with ""
        .fillna({"title": "", "text": ""})
        #restore all indices 
        .reset_index(drop=True)
    )

In [6]:
#for NLP preprocessing and extracting stylometric and sentimental features not required in LDA topic modelling 
num_cols = ["title_len", "text_len", "excl_count", "quest_count", "caps_words"]
raw_combined = df["title"] + " " + df["text"]

df["title_len"] = df["title"].str.len()
df["text_len"] = df["text"].str.len()
df["excl_count"] = raw_combined.str.count(r"!")
df["quest_count"] = raw_combined.str.count(r"\?")
df["caps_words"] = raw_combined.str.count(r"\b[A-Z]{2,}\b")

In [7]:
def clean_url_html(doc: str) -> str:
    
    doc0 = doc.lower()

    #for url regex 
    TLDS = (
    r"com|org|net|gov|edu|mil|int|info|biz|io|co|ly|me|tv|us|uk|ca|au|"
    r"de|fr|in|ru|cn|jp|br|es|it|nl|se|no|ch|news|media|press|online|site|tech|xyz"
    )
    HOST_LOWER = r"[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?"
    PATH_TAIL = r"(?::\d{1,5})?(?:[/?#][^\s<>\"'\)\]]*)?"

    #for html,css tags regex 
    BLOCK_TAG_PAT = re.compile(
        r"<(script|style|iframe)\b[^>]*>.*?</\1>",
        flags=re.IGNORECASE | re.DOTALL,
    )
    HTML_TAG_PAT = re.compile(
        r"<(?:[a-z0-9/]|!--|!doctype)[^>]*>",
        flags=re.IGNORECASE,
    )

    url_pattern = re.compile(
        rf"(?<![\w@])(?:"
        # 1. Links with an explicit protocol or www. (any casing)
        rf"(?:https?|ftps?)://[^\s<>\"']+"
        rf"|www\.{HOST_LOWER}(?:\.{HOST_LOWER})+{PATH_TAIL}"
        # 2. Bare domains without a protocol — lowercase enforced for host and TLD (ignores certificate.It)
        rf"|(?-i:(?:{HOST_LOWER}\.)+(?:{TLDS})){PATH_TAIL}"
        rf")(?!\w)",
        flags=re.IGNORECASE,
    )
    
    docc = url_pattern.sub("", doc0)
    docc = html.unescape(html.unescape(docc))
    docc = BLOCK_TAG_PAT.sub("", docc)
    docc = HTML_TAG_PAT.sub("", docc)

    return docc
    
    #doc0 = doc if isinstance(doc, float) else None 
    #print(doc0)

df["title_clean"] = df["title"].apply(clean_url_html)
df["text_clean"] = df["text"].apply(clean_url_html)

#print(df_raw["title"].dtypes)

In [8]:
#to check for urls in text/title (may delete cell)

TLDS = (
    r"com|org|net|gov|edu|mil|int|info|biz|io|co|ly|me|tv|us|uk|ca|au|"
    r"de|fr|in|ru|cn|jp|br|es|it|nl|se|no|ch|news|media|press|online|site|tech|xyz"
)

HOST_LOWER = r"[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?"
PATH_TAIL = r"(?::\d{1,5})?(?:[/?#][^\s<>\"'\)\]]*)?"

url_pattern = re.compile(
    rf"(?<![\w@])(?:"
    # 1. Links with an explicit protocol or www. (any casing)
    rf"(?:https?|ftps?)://[^\s<>\"']+"
    rf"|www\.{HOST_LOWER}(?:\.{HOST_LOWER})+{PATH_TAIL}"
    # 2. Bare domains without a protocol — lowercase enforced for host and TLD (ignores certificate.It)
    rf"|(?-i:(?:{HOST_LOWER}\.)+(?:{TLDS})){PATH_TAIL}"
    rf")(?!\w)",
    flags=re.IGNORECASE,
)

combined = df["title_clean"].fillna("") + " " + df["text_clean"].fillna("")

df["url"] = combined.apply(url_pattern.findall)

matched = df[df["url"].str.len() > 0]
print(f"Rows with URLs: {len(matched)}")
print(matched["url"].head(20).tolist())

Rows with URLs: 0
[]


In [9]:
#check html tags 
import html 

decoded = combined.apply(lambda s: html.unescape(html.unescape(s)))

html_tags_pat = re.compile(r"<(?:[a-z/]|!--|!doctype)[^>]*>", flags=re.IGNORECASE)
html_entities_pat = re.compile(r"&(?:#\d+|#x[\da-f]+|[a-z][a-z0-9]*);", flags=re.IGNORECASE)

# 2. Check row counts
print("Rows with HTML tags (including hidden/escaped):", decoded.str.contains(html_tags_pat).sum())
print("Rows with leftover HTML entities:", decoded.str.contains(html_entities_pat).sum())

# 3. Extract all unique tag names across the entire dataset (e.g., <p>, <a>, <script>, <iframe>)
all_tags = decoded.apply(lambda s: re.findall(r"</?([a-z0-9]+|!--)", s, flags=re.IGNORECASE))
unique_tags = sorted({tag.lower() for sublist in all_tags for tag in sublist})
print("Unique HTML tag types in corpus:", unique_tags)

Rows with HTML tags (including hidden/escaped): 2
Rows with leftover HTML entities: 5
Unique HTML tag types in corpus: ['25', '7', '72', 'a', 'fsa', 'i', 'jennifer', 'pir', 'she']


In [10]:
#punctuations and numbers remove, since most numbers can disturb cluster fomrations during LDA  
def clean_punct_num(doc: str) -> str:
    #replace ' and - with ""
    doc = re.sub(r"['-]", "", doc)
    #replace anything that isnt alphanum with " "
    doc = re.sub(r"[^a-z0-9\s]", " ", doc)
    #doc = doc.replace("_", " ")
    #replace all numeric characters 
    doc = re.sub(r"\b[0-9]+\b", " ", doc)
    
    return doc 

df["text2"] = df["text_clean"].apply(clean_punct_num)
df["title2"] = df["title_clean"].apply(clean_punct_num)

In [11]:
YEAR_PAT = re.compile(r"\b[0-9]+[a-zA-Z]{1,3}\b")
NUM_PAT = re.compile(r"\b[0-9]+(?:st|nd|rd|th|s)\b")

#removal of words like 21st, 16th, 3rd, 1980s, etc 
df["text3"] = df["text2"].str.replace(NUM_PAT, "", regex=True)
df["title3"] = df["title2"].str.replace(NUM_PAT, "", regex=True)

#lst1 = [item for lst in df["text3"].str.findall(YEAR_PAT) for item in lst if re.search(r"\b([3456]g)\b", item)]
#lst2 = [item for lst in df["title3"].str.findall(YEAR_PAT) for item in lst if re.search(r"\b([3456]g)\b", item)]
#lst1 = [lst for lst in df["text3"].str.findall(YEAR_PAT) if len(lst) != 0]
#lst2 = [lst for lst in df["title3"].str.findall(YEAR_PAT) if len(lst) != 0]

#print(lst1)
#print(lst2)

In [12]:
#motive was to strip the 4g,5g,6g terms from each docs 
def strip_g(doc: str) -> str:
    if not re.search(r"\b([456]g)\b", doc):
        doc = re.sub(YEAR_PAT, "", doc)
    
    return doc 

df["title3"] = df["title3"].apply(strip_g) 
df["text3"] = df["text3"].apply(strip_g) 

#lst1 = [item for lst in df["text3"].str.findall(YEAR_PAT) for item in lst if not re.search(r"\b([3456]g)\b", item)]
#lst2 = [item for lst in df["title3"].str.findall(YEAR_PAT) for item in lst if not re.search(r"\b([3456]g)\b", item)]
lst1 = [lst for lst in df["text3"].str.findall(YEAR_PAT) if len(lst) != 0]
lst2 = [lst for lst in df["title3"].str.findall(YEAR_PAT) if len(lst) != 0]

print(lst1)
print(lst2)

[['5g', '5g', '5g', '5g', '5g', '5g', '5g'], ['5g', '5g', '5g', '4g', '5g'], ['5g', '5g', '5g', '4g'], ['3g', '4g'], ['5g', '4g', '5g'], ['10mm', '15mm', '5g', '5g', '5g'], ['10mm', '15mm', '5g', '5g', '5g'], ['2g', '3g', '4g', '5g', '5g', '5g'], ['3g', '4g', '5g'], ['10mm', '15mm', '5g', '5g', '5g'], ['4g'], ['4g']]
[['5g']]


In [13]:
def remove_stopwrds(doc: str) -> str:
    doc0 = re.sub(r"([0-9]+)([a-zA-Z]{4,})", r"\1 \2", doc)
    doc0 = re.sub(r"\b[0-9]+\b", "", doc0)

    return doc0 

df["text3"] = df["text3"].apply(remove_stopwrds)
df["title3"] = df["title3"].apply(remove_stopwrds)

#lst1 = [lst for lst in df["temp"].str.findall(YEAR_PAT) if len(lst) != 0]
#print(lst1)

In [14]:
from collections import Counter

clean_s = df["text3"].fillna("")
raw_s = (df["title3"].fillna("") + " " + df["text3"].fillna(""))

# 1. Check for ANY leftover punctuation, apostrophes, hyphens, or underscores
leftover_punct = clean_s.str.contains(r"[^a-z0-9\s]", regex=True).sum()
print(f"Rows with leftover punctuation/symbols: {leftover_punct}")
assert leftover_punct == 0, "Found uncleaned punctuation or symbols!"

#'''
# 2. Check for ANY leftover standalone pure numbers
leftover_nums = clean_s.str.contains(r"\b[0-9]+\b", regex=True).sum()
print(f"Rows with leftover standalone pure numbers: {leftover_nums}")
assert leftover_nums == 0, "Found standalone pure numbers!"

# 3. Audit preserved alphanumeric tokens (must contain both at least one letter AND one digit)
alnum_pat = re.compile(r"\b(?=[a-z0-9]*[a-z])(?=[a-z0-9]*[0-9])[a-z0-9]+\b")
alnum_counts = Counter(
    tok for doc in clean_s for tok in alnum_pat.findall(doc)
)
print(f"Unique alphanumeric tokens preserved: {len(alnum_counts)}")
print("Top 20 preserved alphanumerics:", alnum_counts.most_common(20))

# 4. Side-by-side spot-check on rows that originally had contractions, hyphens, numbers & alphanumerics
tricky_mask = (
    raw_s.str.contains(r"\b\w+['’]\w+\b", regex=True) &          # contraction (e.g., don't)
    raw_s.str.contains(r"\b[a-zA-Z]+-[a-zA-Z0-9]+\b", regex=True) & # hyphenated word/code
    raw_s.str.contains(r"\b[0-9]{2,4}\b", regex=True)            # standalone number
)

print("\n--- Side-by-Side Spot Check (3 Tricky Rows) ---")
for idx in df[tricky_mask].head(3).index:
    print(f"\n[Row {idx}] RAW  : {raw_s.loc[idx][:220]}...")
    print(f"[Row {idx}] CLEAN: {clean_s.loc[idx][:220]}...")

#'''

Rows with leftover punctuation/symbols: 0
Rows with leftover standalone pure numbers: 0
Unique alphanumeric tokens preserved: 5470
Top 20 preserved alphanumerics: [('g20', 270), ('f35', 247), ('h1b', 220), ('co2', 184), ('ar15', 149), ('g4s', 146), ('mi6', 141), ('u2009', 130), ('hb2', 105), ('g7', 101), ('mi5', 87), ('g5', 82), ('s400', 79), ('disruptj20', 76), ('ak47', 68), ('ms13', 68), ('post9', 66), ('ww3', 57), ('k12', 56), ('mid1990s', 54)]

--- Side-by-Side Spot Check (3 Tricky Rows) ---


In [16]:
df_clean = df.drop(columns=["text", "title", "text_clean", "title_clean", "url", "text2", "title2"])
df_clean.columns 

Index(['label', 'title_len', 'text_len', 'excl_count', 'quest_count',
       'caps_words', 'text3', 'title3'],
      dtype='str')

In [75]:
#df_clean.to_csv("WELFake2.csv", index=False)

In [76]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

#Count Vectorizer 
- Converts text into fixed-length vectors by frequency of each word in doc, tokens stored as bag of words. 
- generates matrix of shape (num_docs, num_features)

In [77]:
extra_stopwords = {
    # Attribution verbs
    "said", "says", "say", "saying", "told", "according", "reported", "reports",
    "reporting", "stated", "claimed", "claims", "asked", "added", "explained",
    "wrote", "noted", "announced", "confirmed",
    # Media & scraping boilerplate
    "reuters", "getty", "images", "image", "photo", "photos", "video", "videos",
    "watch", "read", "featured", "via", "source", "copyright", "subscribe",
    "click", "follow", "share", "comment", "comments", "twitter", "facebook",
    "youtube", "instagram", "embed", "pic", "post", "posts", "article", "news",
    # Dataset publisher tags
    "21wire", "21stcenturywire", "breitbart", "infowars", "dailycaller", "huffpost", "buzzfeed",
    # Calendar & generic discourse filler
    "monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday",
    "january", "february", "march", "april", "may", "june", "july", "august",
    "september", "october", "november", "december",
    "year", "years", "month", "months", "week", "weeks", "day", "days",
    "time", "times", "today", "yesterday", "ago", "recent", "recently",
    "would", "could", "also", "one", "two", "three", "like", "get", "got",
    "make", "made", "going", "way", "even", "still", "back", "new", "first",
    "last", "many", "much", "well", "use", "used", "using", "want", "know",
    "think", "see", "look", "yearsold",
}

final_stoplist = sorted(stoplist.union(extra_stopwords))

In [80]:
lda_vect = CountVectorizer(
        stop_words=final_stoplist,
        max_df=0.85,
        min_df=20,
        token_pattern=r"\b[a-z0-9]{3,}\b",
        )

In [17]:
'''
df_clean["clean_news"] = (df["title3"] + " " + df["text3"]).str.strip()
#lst = [item for item in df_clean["clean_news"] if item == ""]
#print(lst)
PUBLISHER_LEAK_PAT = re.compile(r"\b(?:reuters|21wire|21stcenturywire|breitbart|infowars)\b", flags=re.I)
df_clean["clean_news"] = df_clean["clean_news"].str.replace(PUBLISHER_LEAK_PAT, " ", regex=True)

df_clean = df_clean[df_clean["clean_news"] != ""].reset_index(drop=True)
'''

In [18]:
df_clean.to_csv("WELFake2.csv", index=False)

In [85]:
lda_ = lda_vect.fit_transform(df_clean["clean_news"])

In [1]:
import pyLDAvis
import pyLDAvis.lda_model
from sklearn.decomposition import LatentDirichletAllocation

#Latent Dirichlet Allocation 
- One important property of the corpus taken is that it incorporates a wide range of topics including national and international politics, economy, investigation, health-care, sports, entertainment, and others.
- To demonstrate topic diveristy the inter-topic distances of the corpus using LDA-based (Latent Dirichlet
Allocation) topic modeling.
- Emprirical estimation of the inter-topic distances dataset was divided into 12 clusters where each cluster represents a topic.
- Top 30 most salient tokens/terms across the dataset post-processing is also evaluated

In [ ]:
lda_model = LatentDirichletAllocation(
    n_components=12,           #num_topics
    learning_method="online", #batch-wise stochastic (EM) update per epoch 
    batch_size=2048, 
    max_iter=15, #num_epochs
    random_state=42,
    n_jobs=-1,
)

doc_topics = lda_model.fit_transform(lda_)

In [ ]:
vocab = lda_vect.get_feature_names_out()
for idx, weights in enumerate(lda_model.components_):
    top_terms = [vocab[i] for i in weights.argsort()[:-13:-1]]
    print(f"Topic {idx + 1:02d}: {', '.join(top_terms)}")

In [ ]:
vis_data = pyLDAvis.lda_model.prepare(
    lda_model,
    lda_,
    lda_vect,
    mds="mmds",
    sort_topics=False,
    n_jobs=1,
)
pyLDAvis.save_html(vis_data, "news_lda_12topics.html")